# FINAL MODEL/PIPELINE FREEZE

In [4]:
# ============================================================
# DAY 22 — FINAL MODEL / PIPELINE FREEZE
# Cell 1: Imports and Setup
# ============================================================

import os
import time
import json
import random
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ------------------------------------------------------------
# TensorFlow information
# ------------------------------------------------------------

print("TensorFlow version:", tf.__version__)

# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------

EXPORT_DIR = "/content/DAY22_FINAL"

os.makedirs(EXPORT_DIR, exist_ok=True)

print("Output directory:", EXPORT_DIR)

TensorFlow version: 2.21.0
Output directory: /content/DAY22_FINAL


In [5]:
# ============================================================
# Cell 2: Mount Google Drive
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

print("Google Drive mounted successfully.")

Mounted at /content/drive
Google Drive mounted successfully.


In [6]:
# ============================================================
# Cell 3: Dataset Path
# ============================================================

DATA_PATH = (
    "/content/drive/MyDrive/"
    "IITG dataset/cleaned_dataset/"
    "processed_dataset_v1/sequences_v1.npz"
)

print("Dataset path:")
print(DATA_PATH)

print("\nFile exists:", os.path.exists(DATA_PATH))

Dataset path:
/content/drive/MyDrive/IITG dataset/cleaned_dataset/processed_dataset_v1/sequences_v1.npz

File exists: True


In [7]:
# ============================================================
# Cell 4: Load Frozen Sequence Dataset
# ============================================================

data = np.load(
    DATA_PATH,
    allow_pickle=True
)

print("Available arrays:")
print(data.files)

X = data["X"]
SoH = data["SoH"]
RUL = data["RUL"]
battery_id = data["battery_id"]
split = data["split"]

print("\nDataset shapes")
print("-" * 40)

print("X          :", X.shape)
print("SoH        :", SoH.shape)
print("RUL        :", RUL.shape)
print("battery_id :", battery_id.shape)
print("split      :", split.shape)

Available arrays:
['X', 'SoH', 'RUL', 'battery_id', 'split']

Dataset shapes
----------------------------------------
X          : (2287, 10, 2)
SoH        : (2287,)
RUL        : (2287,)
battery_id : (2287,)
split      : (2287,)


# Frozen split

In [8]:
# ============================================================
# Cell 5: Verify Frozen Split
# ============================================================

print("Unique split values:")
print(np.unique(split, return_counts=True))

print("\nDetailed split counts")
print("-" * 40)

print("Train      :", np.sum(split == "train"))
print("Validation :", np.sum(split == "validation"))
print("Test       :", np.sum(split == "test"))
print("Total      :", len(split))

Unique split values:
(array(['test', 'train', 'validation'], dtype='<U10'), array([ 392, 1516,  379]))

Detailed split counts
----------------------------------------
Train      : 1516
Validation : 379
Test       : 392
Total      : 2287


In [9]:
# ============================================================
# Cell 6: Create Frozen Train / Validation / Test Sets
# ============================================================

train_idx = split == "train"
val_idx = split == "validation"
test_idx = split == "test"

X_train = X[train_idx]
y_train = SoH[train_idx]

X_val = X[val_idx]
y_val = SoH[val_idx]

X_test = X[test_idx]
y_test = SoH[test_idx]

print("TRAIN")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nVALIDATION")
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("\nTEST")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

TRAIN
X_train: (1516, 10, 2)
y_train: (1516,)

VALIDATION
X_val: (379, 10, 2)
y_val: (379,)

TEST
X_test: (392, 10, 2)
y_test: (392,)


In [23]:
# ============================================================
# DAY 22 — EXACT DAY-19 TCN ARCHITECTURE
# ============================================================

def tcn_residual_block(
    x,
    filters,
    kernel_size,
    dilation_rate,
    dropout
):

    # First causal convolution
    conv1 = tf.keras.layers.Conv1D(
        filters=filters,
        kernel_size=kernel_size,
        padding="causal",
        dilation_rate=dilation_rate,
        activation="relu"
    )(x)

    conv1 = tf.keras.layers.Dropout(
        dropout
    )(conv1)

    # Second causal convolution
    conv2 = tf.keras.layers.Conv1D(
        filters=filters,
        kernel_size=kernel_size,
        padding="causal",
        dilation_rate=dilation_rate,
        activation="relu"
    )(conv1)

    conv2 = tf.keras.layers.Dropout(
        dropout
    )(conv2)

    # Residual connection
    if x.shape[-1] != filters:

        residual = tf.keras.layers.Conv1D(
            filters=filters,
            kernel_size=1,
            padding="same"
        )(x)

    else:

        residual = x

    output = tf.keras.layers.Add()([
        residual,
        conv2
    ])

    output = tf.keras.layers.ReLU()(output)

    return output


def build_tcn_model(
    units=64,
    dropout=0.0,
    learning_rate=0.001
):

    inputs = tf.keras.layers.Input(
        shape=(
            X_train.shape[1],
            X_train.shape[2]
        )
    )

    x = inputs

    # Three TCN residual blocks
    for dilation_rate in [1, 2, 4]:

        x = tcn_residual_block(
            x=x,
            filters=units,
            kernel_size=3,
            dilation_rate=dilation_rate,
            dropout=dropout
        )

    # Take final time step
    x = tf.keras.layers.Lambda(
        lambda tensor: tensor[:, -1, :]
    )(x)

    # Fully connected layer
    x = tf.keras.layers.Dense(
        32,
        activation="relu"
    )(x)

    # Output
    outputs = tf.keras.layers.Dense(1)(x)

    model = tf.keras.Model(
        inputs=inputs,
        outputs=outputs
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss="mse",
        metrics=["mae"]
    )

    return model

In [24]:
# ============================================================
# DAY 22 — BUILD WINNING TCN
# ============================================================

tf.keras.backend.clear_session()

tf.random.set_seed(42)
np.random.seed(42)

FINAL_UNITS = 32
FINAL_DROPOUT = 0.2
FINAL_LEARNING_RATE = 0.00005
FINAL_BATCH_SIZE = 32

final_model = build_tcn_model(
    units=FINAL_UNITS,
    dropout=FINAL_DROPOUT,
    learning_rate=FINAL_LEARNING_RATE
)

final_model.summary()

print("\n" + "=" * 50)
print("FINAL MODEL CONFIGURATION")
print("=" * 50)

print("Units/Filters :", FINAL_UNITS)
print("Dropout       :", FINAL_DROPOUT)
print("Learning Rate :", FINAL_LEARNING_RATE)
print("Batch Size    :", FINAL_BATCH_SIZE)
print("Parameters    :", final_model.count_params())

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 10, 2)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 10, 32)    │        224 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 10, 32)    │          0 │ conv1d[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 10, 32)    │      3,104 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 10, 32)    │         96 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 10, 32)    │          0 │ conv1d_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 10, 32)    │          0 │ conv1d_2[0][0],   │
│                     │                   │            │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 10, 32)    │          0 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 10, 32)    │      3,104 │ re_lu[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 10, 32)    │          0 │ conv1d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 10, 32)    │      3,104 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 10, 32)    │          0 │ conv1d_4[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 10, 32)    │          0 │ re_lu[0][0],      │
│                     │                   │            │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_1 (ReLU)      │ (None, 10, 32)    │          0 │ add_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_5 (Conv1D)   │ (None, 10, 32)    │      3,104 │ re_lu_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 10, 32)    │          0 │ conv1d_5[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_6 (Conv1D)   │ (None, 10, 32)    │      3,104 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 10, 32)    │          0 │ conv1d_6[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 10, 32)    │          0 │ re_lu_1[0][0],    │
│                     │                   │            │ dropout_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_2 (ReLU)      │ (None, 10, 32)    │          0 │ add_2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 32)        │          0 │ re_lu_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │      1,056 │ lambda[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1)         │         33 │ dense[0][0]     

 Total params: 16,929 (66.13 KB)

 Trainable params: 16,929 (66.13 KB)

 Non-trainable params: 0 (0.00 B)


FINAL MODEL CONFIGURATION
Units/Filters : 32
Dropout       : 0.2
Learning Rate : 5e-05
Batch Size    : 32
Parameters    : 16929


# Final Training

In [25]:
# ============================================================
# DAY 22 — FINAL TCN TRAINING
# ============================================================

MAX_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 8

tf.keras.backend.clear_session()

tf.random.set_seed(42)
np.random.seed(42)

final_model = build_tcn_model(
    units=FINAL_UNITS,
    dropout=FINAL_DROPOUT,
    learning_rate=FINAL_LEARNING_RATE
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=EARLY_STOPPING_PATIENCE,
    restore_best_weights=True,
    verbose=1
)

print("Starting final TCN training...")

start_train = time.time()

history = final_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=MAX_EPOCHS,
    batch_size=FINAL_BATCH_SIZE,
    callbacks=[early_stopping],
    verbose=1
)

training_time = time.time() - start_train

print("\nTraining completed.")
print(f"Training time: {training_time:.2f} seconds")

best_epoch = (
    np.argmin(
        history.history["val_loss"]
    ) + 1
)

best_val_loss = np.min(
    history.history["val_loss"]
)

print("\nBest epoch:", best_epoch)
print("Best validation loss:", best_val_loss)

Starting final TCN training...
Epoch 1/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 5s 28ms/step - loss: 6272.3511 - mae: 74.8707 - val_loss: 5294.4819 - val_mae: 68.6035
Epoch 2/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - loss: 6232.4741 - mae: 74.6134 - val_loss: 5244.5054 - val_mae: 68.2521
Epoch 3/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 6192.1104 - mae: 74.3518 - val_loss: 5192.6328 - val_mae: 67.8838
Epoch 4/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 6145.6021 - mae: 74.0479 - val_loss: 5127.9185 - val_mae: 67.4278
Epoch 5/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 6080.2241 - mae: 73.6174 - val_loss: 5010.8330 - val_mae: 66.5893
Epoch 6/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step - loss: 5943.5420 - mae: 72.6985 - val_loss: 4741.8350 - val_mae: 64.6247
Epoch 7/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 5661.6523 - mae: 70.7852 - val_loss: 4218.5825 - val_mae: 60.9169
Epoch 8/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 5081.2017 - mae: 67.0318 - val_loss

In [26]:
# ============================================================
# DAY 22 — VALIDATION CHECK
# ============================================================

y_val_pred = final_model.predict(
    X_val,
    verbose=0
).ravel()

val_mse = mean_squared_error(
    y_val,
    y_val_pred
)

val_rmse = np.sqrt(val_mse)

val_mae = mean_absolute_error(
    y_val,
    y_val_pred
)

val_r2 = r2_score(
    y_val,
    y_val_pred
)

print("\n" + "=" * 55)
print("DAY 22 VALIDATION RESULTS")
print("=" * 55)

print(f"MAE  : {val_mae:.6f}")
print(f"RMSE : {val_rmse:.6f}")
print(f"R²   : {val_r2:.6f}")

print("\nDay-19 reference:")
print("MAE  : 14.343437")
print("RMSE : 21.880802")
print("R²   : 0.181139")


DAY 22 VALIDATION RESULTS
MAE  : 16.875757
RMSE : 22.857713
R²   : 0.106387

Day-19 reference:
MAE  : 14.343437
RMSE : 21.880802
R²   : 0.181139
